In [2]:
%pip install pdfplumber pymupdf

   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
    --------------------------------------- 0.3/19.8 MB ? eta -:--:--
   -- ------------------------------------- 1.0/19.8 MB 2.1 MB/s eta 0:00:09
   ---- ----------------------------------- 2.1/19.8 MB 3.3 MB/s eta 0:00:06
   ------- -------------------------------- 3.7/19.8 MB 4.4 MB/s eta 0:00:04
   ------------- -------------------------- 6.6/19.8 MB 6.4 MB/s eta 0:00:03
   -------------------- ------------------- 10.0/19.8 MB 8.2 MB/s eta 0:00:02
   -------------------------- ------------- 13.1/19.8 MB 9.2 MB/s eta 0:00:01
   ---------------------------------- ----- 17.0/19.8 MB 10.4 MB/s eta 0:00:01
   ---------------------------------------- 19.8/19.8 MB 11.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import re
from gensim.utils import simple_preprocess
from textblob import TextBlob
import nltk
from langdetect import detect
import pdfplumber
import fitz  # PyMuPDF
from pathlib import Path



class TextProcessor:
    def __init__(self, texts, directory=None):
        self.texts = texts
        self.directory = directory

    # Load text files
    @classmethod
    def load_text_files(cls, directory):
        text_data = [] # sample: ["Text of document 1", "Text of document 2", ...]
        file_paths = []

        # Use pathlib.Path to dynamically list all .txt files in directory
        for filepath in Path(directory).rglob("A*.pdf"):
            with pdfplumber.open(filepath) as pdf:
                text = ""
                for page in pdf.pages:
                    text += page.extract_text()
                text_data.append(text)
                file_paths.append(filepath)
        print("Loaded {} documents from {}".format(len(file_paths), directory))
        return cls(text_data, directory), file_paths

    def clean_text(self, text):
        # Remove special characters and digits
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        # Convert to lowercase
        text = text.lower()
        print("Cleaned text: ", text[:100])  # Print first 100 characters of cleaned text
        return text

    def detect_language(self, text):
        try:
            print("Detecting language for text: ", text[:100])  # Print first 100 characters of text for language detection
            return detect(text)
        except:
            return "unknown"

    def tokenize_text(self, text):
        from nltk.tokenize import word_tokenize
        print("Tokenizing text: ", text[:100])  # Print first 100 characters of text for tokenization
        return word_tokenize(text)

    def chunk_text(self, text, chunk_size=100):
        words = self.tokenize_text(text)
        chunked_texts = []
        for i in range(0, len(words), chunk_size):
            chunked_texts.append(' '.join(words[i:i + chunk_size]))
        print("Chunked text into {} chunks.".format(len(chunked_texts)))  # Print number of chunks created
        return chunked_texts
        # return [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

    def correct_spelling(self, text):
        blob = TextBlob(text)
        print("Correcting spelling for text: ", text[:100])  # Print first 100 characters of text for spelling correction
        return str(blob.correct())

    def analyze_sentiment(self, text):
        blob = TextBlob(text)
        return blob.sentiment.polarity, blob.sentiment.subjectivity

    def process_texts(self):
        processed_data = []
        cleaned_texts = []
        for text in self.texts:
            cleaned_text = self.clean_text(text)

            print("Cleaned text: ", cleaned_text[:100])  # Print first 100 characters of cleaned text
            # corrected_text = self.correct_spelling(cleaned_text)

            # print("Corrected text: ", corrected_text[:100])  # Print first 100 characters of corrected text
            language = self.detect_language(cleaned_text)
            chunked_texts = self.chunk_text(cleaned_text) # sample: ["Chunk 1 text", "Chunk 2 text", ...]
            print("Detected language: ", language)  # Print detected language
            polarity, subjectivity = self.analyze_sentiment(cleaned_text)
            processed_data.append({
                "original_text": text,
                "cleaned_text": cleaned_text,
                # "corrected_text": corrected_text,
                'chunked_texts': chunked_texts,
                "language": language,
                "polarity": polarity,
                "subjectivity": subjectivity
            })
            cleaned_texts.append(cleaned_text)
        print("Processed {} texts.".format(len(processed_data)))  # Print number of processed texts
        return processed_data, cleaned_texts

In [2]:
# Data directory
DATA_DIR = Path("./Procedures_and_Standards_vehicles_explosives")
text_processor_object, file_path = TextProcessor.load_text_files(DATA_DIR)
processed_data, cleaned_text = text_processor_object.process_texts()

Loaded 2 documents from Procedures_and_Standards_vehicles_explosives
Cleaned text:  acme mine tyre wheel and rim management
procedure
dummy document for rag testing  not for operationa
Cleaned text:  acme mine tyre wheel and rim management
procedure
dummy document for rag testing  not for operationa
Detecting language for text:  acme mine tyre wheel and rim management
procedure
dummy document for rag testing  not for operationa
Tokenizing text:  acme mine tyre wheel and rim management
procedure
dummy document for rag testing  not for operationa
Chunked text into 14 chunks.
Detected language:  en
Cleaned text:  acme mine
safety health and management
system
comprehensive document set
mine guard ai shms document
Cleaned text:  acme mine
safety health and management
system
comprehensive document set
mine guard ai shms document
Detecting language for text:  acme mine
safety health and management
system
comprehensive document set
mine guard ai shms document
Tokenizing text:  acme mine
safety

In [3]:
# View chunked text of the first document
processed_data[0]['chunked_texts']  # Display the first processed text data for verification

['acme mine tyre wheel and rim management procedure dummy document for rag testing not for operational use document control item detail document id acmeshmspltpr title tyre wheel and rim management procedure version draft effective date july review date july owner maintenance superintendent sponsor site senior executive sse approver operations manager controlled copy electronic copy in shms portal purpose to define the minimum requirements for the safe lifecycle management of tyres wheels and rims at acme mine including selection inspection handling fitting inflation operation repair and disposal to eliminate or minimise the risk of tyre and wheel related incidents eg explosions',
 'separations roll aways and dropped loads scope this procedure applies to all acme mine personnel and contractors involved with or potentially exposed to heavy vehicle and light vehicle tyres wheels and rims on site surface operations workshops tyre bays laydown areas and haul roads references recognised sta

In [4]:
text_processor_object.texts # Display the first cleaned text for verification

['ACME Mine Tyre, Wheel and Rim Management\nProcedure\n(Dummy document for RAG testing – not for operational use)\n1. Document Control\nItem Detail\nDocument ID ACME-SHMS-PLT-PR-013\nTitle Tyre, Wheel and Rim Management Procedure\nVersion 0.1 (Draft)\nEffective Date 27 July 2025\nReview Date 27 July 2027\nOwner Maintenance Superintendent\nSponsor Site Senior Executive (SSE)\nApprover Operations Manager\nControlled Copy Electronic copy in SHMS Portal\n2. Purpose\nTo define the minimum requirements for the safe lifecycle management of tyres, wheels and rims at\nACME Mine, including selection, inspection, handling, fitting, inflation, operation, repair and disposal, to\neliminate or minimise the risk of tyre and wheel related incidents (e.g. explosions, separations, roll-\naways and dropped loads).\n3. Scope\nThis procedure applies to all ACME Mine personnel and contractors involved with or potentially exposed\nto heavy vehicle and light vehicle tyres, wheels and rims on site (surface ope

In [5]:
# Aggregate all chunked texts from the processed data
all_chunked_texts = []
for data in processed_data:
    all_chunked_texts.extend(data['chunked_texts'])

all_chunked_texts[:5]  # Display the first 5 chunked texts for verification

['acme mine tyre wheel and rim management procedure dummy document for rag testing not for operational use document control item detail document id acmeshmspltpr title tyre wheel and rim management procedure version draft effective date july review date july owner maintenance superintendent sponsor site senior executive sse approver operations manager controlled copy electronic copy in shms portal purpose to define the minimum requirements for the safe lifecycle management of tyres wheels and rims at acme mine including selection inspection handling fitting inflation operation repair and disposal to eliminate or minimise the risk of tyre and wheel related incidents eg explosions',
 'separations roll aways and dropped loads scope this procedure applies to all acme mine personnel and contractors involved with or potentially exposed to heavy vehicle and light vehicle tyres wheels and rims on site surface operations workshops tyre bays laydown areas and haul roads references recognised sta

In [6]:
from transformers import TFAutoModel, AutoTokenizer
import tensorflow as tf
import numpy as np

# Verify that TensorFlow detects the GPU
print("Available devices:", tf.config.list_physical_devices('GPU'))

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = TFAutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

def get_embeddings_in_batch(texts, batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="tf")
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state[:, 0, :]  # Use the [CLS] token representation
        all_embeddings.append(embeddings.numpy())
    return np.vstack(all_embeddings) # Stack all embeddings vertically to create a single array

# Get embeddings for all chunked texts
embeddings = get_embeddings_in_batch(all_chunked_texts)
print("Generated embeddings for {} chunked texts.".format(embeddings.shape[0]))  # Print number of embeddings generated
embeddings[0]  # Display the first embedding for verification

Available devices: []




Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['embeddings.position_ids']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions without further training.


Generated embeddings for 104 chunked texts.


array([-4.10392210e-02,  4.11900729e-02, -2.23861516e-01, -8.89505073e-02,
       -2.20267311e-01, -2.71111786e-01, -1.35497868e-01,  2.63487190e-01,
       -3.71831745e-01,  2.14154691e-01,  3.83231133e-01,  2.97923476e-01,
        6.43669844e-01, -1.80761099e-01, -1.81914076e-01,  3.74267370e-01,
        1.48991168e-01, -2.88619101e-02, -2.32366905e-01, -4.16546375e-01,
       -7.84674436e-02,  5.23781553e-02,  1.03469133e-01,  2.02422440e-02,
       -2.45889664e-01,  7.92321488e-02, -1.65786743e-01,  6.15154386e-01,
       -3.34914140e-02, -6.92407846e-01,  6.91280216e-02,  4.03082937e-01,
       -1.22315250e-02, -3.39983076e-01,  2.75432587e-01, -1.15438476e-01,
       -2.33422503e-01, -1.13056175e-01,  7.43516162e-02, -3.93905669e-01,
        3.21425527e-01, -3.46762031e-01, -5.25403433e-02, -1.62850305e-01,
       -1.13259040e-01, -1.77539900e-01, -7.85541087e-02, -2.83085287e-01,
       -1.42176710e-02,  1.85520783e-01, -2.23019347e-03,  2.42502004e-01,
        3.97097439e-01, -

In [7]:
# Embeddings shape
print("Embeddings shape:", embeddings.shape)  # Print the shape of the embeddings array

Embeddings shape: (104, 384)


In [8]:
import faiss
import numpy as np

# Define the dimension of embeddings
dimension = embeddings.shape[1]  # Embedding size from MiniLM model
index = faiss.IndexFlatL2(dimension)

index.add(embeddings)  # Add embeddings to the FAISS index
print("FAISS index created with {} embeddings.".format(index.ntotal))  # Print number of embeddings added to the FAISS index

FAISS index created with 104 embeddings.


In [9]:

def get_query_embedding(query):
    inputs = tokenizer([query], padding=True, truncation=True, return_tensors="tf")
    outputs = model(**inputs)
    query_embedding = outputs.last_hidden_state[:, 0, :].numpy()  # Use the [CLS] token representation
    return query_embedding

In [10]:
# Search index for the most similar chunks to a query
query = "What are the safety procedures for handling explosives?"
embedding = get_query_embedding(query)
k = 5  # Number of nearest neighbors to retrieve
distances, indices = index.search(embedding, k)  # Search the FAISS index for nearest neighbors 
print("Query:", query)
print("Nearest neighbors (indices):", indices)
print("Distances to nearest neighbors:", distances)

Query: What are the safety procedures for handling explosives?
Nearest neighbors (indices): [[43 39 32 37 38]]
Distances to nearest neighbors: [[16.301369 16.715172 18.717628 18.826485 19.538471]]


In [11]:
# Print the actual chunked texts corresponding to the nearest neighbor indices
for idx in indices[0]:
    print("Chunked text:", all_chunked_texts[idx])

Chunked text: explosives storage areas fuel storage and dispensing areas and administration buildings definitions fire explosion an uncontrolled combustion event a sudden and violent release of energy involving flammable or combustible resulting from rapid chemical reaction materials pressure buildup or ignition of explosive atmospheres emergency response team ert incident controller a trained group of personnel responsible the person responsible for overall for responding to site emergencies management and coordination of including fires and explosions emergency response activities during an incident roles responsibilities site senior executive sse emergency response coordinator ensureadequateresourcesareavailable maintainemergencyresponseequipmentfirst aid response procedure emergency management group effective date review period years
Chunked text: competent personnel shall handle transport and use explosives transport requirements explosives shall be transported in approved clearly

In [12]:
def retrieve_relevant_chunks(query, k=5):
    embedding = get_query_embedding(query)
    distances, indices = index.search(embedding, k)
    relevant_chunks = [all_chunked_texts[idx] for idx in indices[0]]
    return relevant_chunks, distances[0]

In [13]:
%pip install anthropic dotenv

Note: you may need to restart the kernel to use updated packages.


# Generate Response

In [14]:
from dotenv import load_dotenv
import anthropic
import os


load_dotenv()  # Loads .env file
api_key = os.getenv("API_KEY")

client = anthropic.Anthropic(api_key=api_key)

def generate_response(query, context, max_new_tokens=100):
    message = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=max_new_tokens,
        messages=[{
            "role": "user",
            "content": f"User query: {query}\n\nContext:\n{context}\n\nAnswer:"
        }]
    )
    return message.content[0].text

In [15]:
# Retrieve relevant context and distances for the query
context, distances = retrieve_relevant_chunks(query, k=5)
print("Retrieved context:", context)  # Print the retrieved context
print("Distances:", distances)  # Print the distances

# Query the Generative LLM
response = generate_response(query, context)
print("Response from Generative LLM:", response)  # Print the response from the Generative LLM

Retrieved context: ['explosives storage areas fuel storage and dispensing areas and administration buildings definitions fire explosion an uncontrolled combustion event a sudden and violent release of energy involving flammable or combustible resulting from rapid chemical reaction materials pressure buildup or ignition of explosive atmospheres emergency response team ert incident controller a trained group of personnel responsible the person responsible for overall for responding to site emergencies management and coordination of including fires and explosions emergency response activities during an incident roles responsibilities site senior executive sse emergency response coordinator ensureadequateresourcesareavailable maintainemergencyresponseequipmentfirst aid response procedure emergency management group effective date review period years', 'competent personnel shall handle transport and use explosives transport requirements explosives shall be transported in approved clearly mar